## Catalog and Schema
This dimension table will be used to join the two tables from Entsoe and with synthetic sensor data. It will be also used to present the Slowly Changing Dimension 2.

In [0]:
# Configuration
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.combobox("bronze_schema", "bronze", ["bronze", "gabrielajaniszews786_bronze"], "Bronze schema")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
CATALOG       = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")


In [0]:
# Creating Silver Schema
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SILVER_SCHEMA}")

## Creating a Silver Table

In [0]:
spark.sql(f"""
          CREATE OR REPLACE TABLE {CATALOG}.{SILVER_SCHEMA}.dim_datacenter
          (
            dc_key BIGINT GENERATED ALWAYS AS IDENTITY,
            site_id STRING NOT NULL,
            site_name STRING,
            country STRING,
            bidding_zone STRING,
            valid_from TIMESTAMP NOT NULL,
            valid_to TIMESTAMP,
            is_current BOOLEAN NOT NULL,
            CONSTRAINT pk_dim_datacenter PRIMARY KEY (dc_key))
            """)

In [0]:
# Seeding the dimension with the current version of EVERY site, straight from the source.
# One row per site, is_current = true, open-ended (valid_to = NULL).
spark.sql(f"""
INSERT INTO {CATALOG}.{SILVER_SCHEMA}.dim_datacenter
    (site_id, site_name, country, bidding_zone, valid_from, valid_to, is_current)
SELECT DISTINCT
    site_id,
    site_name,
    country,
    bidding_zone,
    current_timestamp()       AS valid_from,
    CAST(NULL AS TIMESTAMP)    AS valid_to,
    true                       AS is_current
FROM {CATALOG}.{BRONZE_SCHEMA}.sensor_data
""")

display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.dim_datacenter ORDER BY site_id"))

## Inserting New Site Name
In this scenario, the company is building a second data center near Warsaw. Therefore the existing one will get a new site_name "Warsaw DC 1" to differentiate it from the new site DC-PL-02 that after commissioning will be called "Warsaw DC 2".

In a real pipeline this change would arrive from the source (the system suddenly reports DC-PL-01 under a new name). Here we simulate it with a single row and register it as a temporary view so MERGE has something to read.

In [0]:
# The incoming change from the source: DC-PL-01 is renamed to "Warsaw DC 1"
updates = spark.createDataFrame(
    [("DC-PL-01", "Warsaw DC 1", "PL", "PL")],
    ["site_id", "site_name", "country", "bidding_zone"]
)
updates.createOrReplaceTempView("dim_updates")

A single MERGE matches a source row to one target row and runs one action, so we duplicate each change. One copy carries the key (mergeKey = site_id) and will hit the current row so we can close it. The other carries NULL, hits nothing, and lands as a fresh version. The JOIN at the end passes only rows where the name actually changed, which is what makes a re-run safe.

In [0]:
# Two copies of each change:
#  - mergeKey = site_id  -> MATCHES the current row so we can CLOSE it
#  - mergeKey = NULL      -> never matches -> gets INSERTED as the new version
# The JOIN keeps only rows that actually changed (idempotent on re-run).
staged = spark.sql(f"""
    SELECT u.site_id AS mergeKey, u.*
    FROM dim_updates AS u
    UNION ALL
    SELECT NULL AS mergeKey, u.*
    FROM dim_updates AS u
    JOIN {CATALOG}.{SILVER_SCHEMA}.dim_datacenter AS d
      ON d.site_id = u.site_id
     AND d.is_current = true
     AND d.site_name <> u.site_name
""")
staged.createOrReplaceTempView("dim_scd2_staged")

One MERGE with two branches. When it hits the current row and the name differs, it closes the old version: sets is_current = false and writes valid_to. When it hits nothing (the NULL copy), it inserts the new version as current. Both branches fire in a single pass.

In [0]:
# One MERGE, two effects:
#  MATCHED + name changed -> close the old version
#  NOT MATCHED            -> open the new version
spark.sql(f"""
MERGE INTO {CATALOG}.{SILVER_SCHEMA}.dim_datacenter AS d
USING dim_scd2_staged AS s
  ON d.site_id = s.mergeKey AND d.is_current = true
WHEN MATCHED AND d.site_name <> s.site_name THEN
  UPDATE SET d.is_current = false,
             d.valid_to   = current_timestamp()
WHEN NOT MATCHED THEN
  INSERT (site_id, site_name, country, bidding_zone, valid_from, valid_to, is_current)
  VALUES (s.site_id, s.site_name, s.country, s.bidding_zone, current_timestamp(), NULL, true)
""")

We look at what the MERGE did to DC-PL-01. There should be two rows: the old "Warsaw DC" with valid_to filled in and is_current = false, and the new "Warsaw 1" with valid_to = NULL and is_current = true. That is the history "generated" by SCD2.

In [0]:
display(spark.sql(f"""
    SELECT dc_key, site_id, site_name, valid_from, valid_to, is_current
    FROM {CATALOG}.{SILVER_SCHEMA}.dim_datacenter
    WHERE site_id = 'DC-PL-01'
    ORDER BY valid_from
"""))

This is the manual version of SCD2, written so the mechanics are visible. In production you usually wrap it in a single apply_scd2(...) function and call it in one line per dimension, or hand it to the engine: Lakeflow / Delta Live Tables has APPLY CHANGES ... STORED AS SCD TYPE 2, where you declare only the key and the ordering column and the UNION trick happens under the hood.